# 01 Profile

#### this notebook is to run the profiling functions we created so we can diagnose any inconsistencies within our data

In [2]:
from src.profiler import run_profile

In [3]:
# execute the run_profile to generate the log files for all csv data
run_profile() # output of this is saved in /output/logs/

(                               file   start_s  ...  max_gap_s  gap_time_s
 0                          hv_power  0.011848  ...   6.268515  182.346495
 1                  imu_angular_rate  0.001332  ...   6.289916   28.896991
 2                  vehicle_position  2.188464  ...   6.215764   23.079007
 3                     vehicle_speed  2.188702  ...   6.215893   23.080508
 4             pedal_brakes_pressure  0.008073  ...   6.293122   35.408497
 5                       steer_angle  0.004815  ...   6.297040   39.091449
 6   vehicle_curvilinear_coordinates  2.188464  ...   6.215764   23.079007
 7                    pedal_throttle  0.008140  ...   6.292822   46.778663
 8                  imu_acceleration  0.002541  ...   6.290568   29.018449
 9            front_angular_velocity  0.000000  ...   6.292865   26.714684
 10        inv_r_id_a8_n_actual_filt  0.008493  ...   6.229199   23.719827
 11        inv_l_id_a8_n_actual_filt  0.008242  ...   6.230026   23.685588
 12            inv_r_id_2

In [5]:
import pandas as pd
from src.config import OUT_LOGS

In [12]:
profile_file = pd.read_csv(OUT_LOGS / 'profile_files.csv')
profile_file

,file,start_s,end_s,duration_s,n_samples,is_sorted,n_duplicates,rate_hz,n_gaps,max_gap_s,gap_time_s
0,hv_power,0.011848,2206.954179,2206.942331,35826,True,0,19.997600,1289,6.268515,182.346495
1,imu_angular_rate,0.001332,2206.967405,2206.966073,489829,True,0,223.413762,277,6.289916,28.896991
2,vehicle_position,2.188464,2206.960475,2204.772011,212334,True,0,97.399435,72,6.215764,23.079007
3,vehicle_speed,2.188702,2206.960777,2204.772075,212359,True,0,97.399435,72,6.215893,23.080508
4,pedal_brakes_pressure,0.008073,2206.966978,2206.958905,207940,True,0,99.492588,487,6.293122,35.408497
5,steer_angle,0.004815,2206.962417,2206.957602,207179,True,0,99.542106,653,6.297040,39.091449
6,vehicle_curvilinear_coordinates,2.188464,2206.960475,2204.772011,212334,True,0,97.399435,72,6.215764,23.079007
7,pedal_throttle,0.008140,2206.967259,2206.959119,203103,True,0,99.443119,956,6.292822,46.778663
8,imu_acceleration,0.002541,2206.968685,2206.966144,489829,True,0,223.413762,285,6.290568,29.018449
9,front_angular_velocity,0.000000,2206.962362,2206.962362,213933,True,0,99.631364,128,6.292865,26.714684


**Notes**
* notice some late starts (2s) in vehicle_position, vehicle_speed, vehicle_curvilinear_coordinates and GPS_NAV_PVT by approx. 2s
* the end_s is all the same.
* duration is ~2206 +/- 2s for the previously mentioned data
* n_samples reflects the different sample rate
* all data is sorted (which is good)
* there are no duplicated rows
* sampling rates are 20Hz,  ~100Hz, ~223Hz, and ~7Hz
* there are gaps in the data for the gap_factor = 3
* max gap duration is ~6s accross all data
* hv_power has a cumulative gap of ~182s / 2207 (~8% of data) compared to an avg of ~25-40s for the rest (1%-2% of data)
* the inverters and estimators ile all have ~70gaps and ~23s lost
* vehicle_position and vehicle_curvilinear_coordinates are identical (start, end, count, gaps)

In [9]:
profile_columns = pd.read_csv(OUT_LOGS / 'profile_columns.csv')
profile_columns

,file,column,n_nan,min,max,n_unique,longest_run,longest_run_s,longest_run_start_s
0,hv_power,power,0,-5.630580e-01,6.308843e+01,9176,145,9.599642,1922.060700
1,imu_angular_rate,x,0,-2.126000e+01,3.227000e+01,3155,6,0.022411,983.115400
2,imu_angular_rate,y,0,-3.373000e+01,2.269000e+01,3118,5,0.017908,1067.320481
3,imu_angular_rate,z,0,-8.960000e+01,1.163000e+02,17619,6,0.022394,57.353199
4,vehicle_position,x,0,-4.340124e+02,-2.176700e+02,7089,639,6.561277,1873.054750
5,vehicle_position,y,0,-5.796903e+01,1.142443e+02,5641,729,7.484125,1897.333112
6,vehicle_position,heading,0,-3.141590e+00,3.141494e+00,56208,10,0.093161,19.607176
7,vehicle_speed,u,0,0.000000e+00,2.374060e+01,29170,22288,230.706131,1976.254646
8,vehicle_speed,v,0,-2.854353e+00,3.778439e+00,8851,7,0.061039,1196.456935
9,pedal_brakes_pressure,front,0,0.000000e+00,2.267185e+01,1019,21047,233.008006,1973.958972


**Notes**
* no NaN values
* some columns are redundent as they have only one data such as imu_temperature (value = 0), headVeh, magDec, and others.
* behicle speed u had a longest run of 230s starting at s 1976 so it probably is when the car came to a stop at the end.
* front and read brake pressure, steer angle, and s also had a very long run starting at 1973s so also probably the car is parked at the end of the race.
* throttle freezes aarlier than the rest: constant run start at 1827s which is abt 150s before the car stops at 1976s, so either coasting or being pushed or the sensor froze (plot against speed to find out)
* position estimate (s) keeps changing while parked, it's the same for 144s from 1970 then changes again around 2115s with u = 0
* min wheel speed is negative (min = -0.54), inverter min is -1.4. offset or noise, need to check in standstill
* negative value for power means energy is flowing into the battery, min value of -0.56 either very little regen or noise -> check when it occurs to know if it is regen or just offset or noise


#### some findings from this notebook

1. constant columns that we can drop: imu_temperature, headVeh, magDec and the GPS date fields (year,month,day), valid,flags2,fixType (3 = 3D fix)
2. late starts for some estimators and GPS